# LLM Experiment

## 0. Imports

In [1]:
import sys
import os
import pandas as pd

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from src.semantic import load_faiss
from src.rag_pipeline import (
    build_hybrid_retriever,
    build_context,
    build_prompt_template,
    build_llm,
)
from src.prompts import SYSTEM_PROMPT_V3

## 1. Define queries

We reuse five queries from the previous experiment to compare outputs of two language models. 

In [2]:
_, docs = load_faiss("../semantic_index")

print(f"Loaded {len(docs)} documents")

test_queries = [
    "refrigerator water filter",
    "something to keep drinks cold in a dorm room",
    "energy-efficient refrigerator for a family of four",
    "best dishwasher for a small apartment under $800",
    "reliable stove for frequent home cooking that is easy to clean",
]

Loaded 19999 documents


## 2. Create context retriever

In [3]:
K = 5
RRF_K = 60

hybrid_retriever = build_hybrid_retriever(
    docs=docs,
    k=K,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    rrf_k=RRF_K,
)

prompt_template = build_prompt_template(SYSTEM_PROMPT_V3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3. Build language models for the RAG pipeline

We compare the baseline model `meta-llama/Meta-Llama-3-8B-Instruct` to `Qwen/Qwen2.5-7B-Instruct`. The later is has large capacity and which is released as an instruction-tuned language model that excels in coding, mathematics, and structured data generation

In [4]:
MODEL_A = "meta-llama/Meta-Llama-3-8B-Instruct"
MODEL_B = "Qwen/Qwen2.5-7B-Instruct"

llm_a = build_llm(repo_id=MODEL_A)
llm_b = build_llm(repo_id=MODEL_B)

## 4. Test language models on the same queries

We pass the above five queries to both models. Note that due to the nature of LLMs, outputs will change if you execute the below cells again.

In [5]:
comparison_rows = []

for query in test_queries:
    retrieved_docs = hybrid_retriever.invoke(query)
    context = build_context(retrieved_docs)

    prompt_value = prompt_template.invoke({
        "context": context,
        "question": query
    })

    answer_a = llm_a.invoke(prompt_value)
    answer_b = llm_b.invoke(prompt_value)

    text_a = answer_a.content if hasattr(
        answer_a, "content") else str(answer_a)
    text_b = answer_b.content if hasattr(
        answer_b, "content") else str(answer_b)

    comparison_rows.append({
        "query": query,
        "context": context,
        "model_a": MODEL_A,
        "answer_a": text_a,
        "model_b": MODEL_B,
        "answer_b": text_b,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,query,context,model_a,answer_a,model_b,answer_b
0,refrigerator water filter,Source: metadata\nASIN: B01N21JIYF\nMain Categ...,meta-llama/Meta-Llama-3-8B-Instruct,"Based on the provided context, here are 3 bull...",Qwen/Qwen2.5-7B-Instruct,"- **Best Match(es)**: ASIN B01N21JIYF, ASIN B0..."
1,something to keep drinks cold in a dorm room,Source: metadata\nASIN: B01FQGWN34\nMain Categ...,meta-llama/Meta-Llama-3-8B-Instruct,* **Best Matches:** \n - B01FQGWN34 (Haier 1....,Qwen/Qwen2.5-7B-Instruct,- **Best Match:** Haier 1.7 cu ft Refrigerator...
2,energy-efficient refrigerator for a family of ...,Source: review\nASIN: B07CG6YZCZ\nRating: 5.0/...,meta-llama/Meta-Llama-3-8B-Instruct,"Based on the provided context, the best match ...",Qwen/Qwen2.5-7B-Instruct,- **Best match:** Frestec Mini Fridge with Fre...
3,best dishwasher for a small apartment under $800,Source: metadata\nASIN: B07GBHS24F\nMain Categ...,meta-llama/Meta-Llama-3-8B-Instruct,"Based on the provided context, the following a...",Qwen/Qwen2.5-7B-Instruct,- **Best Match:** DELLA Compact Mini Dishwashe...
4,reliable stove for frequent home cooking that ...,Source: metadata\nASIN: B0B7RNTHPJ\nMain Categ...,meta-llama/Meta-Llama-3-8B-Instruct,* **Best Match:** \n - ASIN: B08CY9BCS6 (Chef...,Qwen/Qwen2.5-7B-Instruct,- **Best match:** Dreyoo aluminum burner bibs ...


In [6]:
for _, row in comparison_df.iterrows():
    print("=" * 60)
    print("QUERY:")
    print(row["query"])
    print()
    print("MODEL A:")
    print(row["model_a"])
    print(row["answer_a"])
    print()
    print("MODEL B:")
    print(row["model_b"])
    print(row["answer_b"])
    print()

QUERY:
refrigerator water filter

MODEL A:
meta-llama/Meta-Llama-3-8B-Instruct
Based on the provided context, here are 3 bullet points covering the best match(es) for a refrigerator water filter:

* The best match for a refrigerator water filter is the product with metadata ASIN: B01N21JIYF. This product is described as an LG Refrigerator Water Filter Replacement, indicating it is suitable for an LG refrigerator.
* This product is also relevant because it is specifically stated to be compatible with the LG Water Filter LT700P, among other models. This suggests it may be a suitable replacement for existing LG refrigerator filters.
* An important limitation or uncertainty is that the provided reviews do not explicitly mention the product with ASIN: B01N21JIYF. However, the other reviews (with ASINs B0045LLC7K, B0169T71UW, and B01CC7IVF4) are all positive and mention being happy with the water filter for their refrigerator, indicating a general satisfaction with refrigerator water filters